# DREME Commit 6 · Tidy3D SOI comparison

Run **0 → 1** locally. Use the new source-aligned output root. The earlier baseline tasks used a shifted source; prepare and inspect uniform before adding moving. Steps **2 → 3** are user-operated, one case at a time, with a separate cost approval. Step **4** reads existing local HDF5. A fresh notebook has both cloud selectors disabled.

Use Python **env310** (Python 3.10, Tidy3D 2.12.0). See `TIDY3D_VALIDATION.md` for credentials, outputs and interpretation. Confirm the two editable geometries and FDTD starting settings before uploading.

Completed source-aligned runs are reviewed in section **5** after section **0**, using local files. See `../results/coupled_waveguide/README.md` for the recorded discrepancies and open accuracy criteria.


## 0 — Environment and settings

The installed defaults use the existing 33-gap, 24-mode DREME dataset and the aligned 250 nm reference fields. The 17-gap dataset estimates sensitivity to the gap grid. The lengths, FDTD mesh, source offset and run time are initial settings for review. Set a fresh `OUTPUT_ROOT` when changing settings after the first preflight.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

REPO = Path(r"D:\data\SongJaesung\git\dataset-based-eme-coupled-waveguide")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import pandas as pd
from IPython.display import display
import tidy3d as td
from examples.coupled_waveguide.tidy3d_comparison import (
    prepare_case, plot_case, upload_and_estimate, run_or_download, compare_saved,
)
assert td.__version__ == "2.12.0", td.__version__
print(sys.executable, "| Tidy3D", td.__version__)

DATA_ROOT = Path.home() / "DREME" / "coupled_waveguide_runs"
DATASET = DATA_ROOT / "soi220_power_debug_33nodes_24modes_mesh352x208_20261106_205757"
COARSE_DATASET = DATA_ROOT / "soi220_power_debug_17nodes_24modes_mesh352x208_20261106_204648"
REFERENCE_FIELDS = DATA_ROOT / "tidy3d_c6_33node_reference" / "reference_250nm.npz"
OUTPUT_ROOT = DATA_ROOT / "tidy3d_c6_source_aligned_ppw22"
LENGTHS_UM = (10.0, 5.0, 5.0, 5.0)  # uniform, approach, hold, departure
FDTD_MIN_STEPS_PER_WVL = 22  # Minimum at the comparison wavelength, inside silicon.
with np.load(REFERENCE_FIELDS, allow_pickle=False) as ref:
    source_metadata = json.loads(str(ref["metadata"]))
FDTD_MESH_UM = source_metadata["wavelength_m"] * 1e6 / (
    source_metadata["material_indices"]["si"][0] * FDTD_MIN_STEPS_PER_WVL
)
print(f"Uniform FDTD mesh: at most {FDTD_MESH_UM * 1000:.3f} nm per cell")
FDTD_RUN_TIME_PS = 3.0
SOURCE_X_UM = -0.125  # x=0 is the DREME input reference plane
LEAD_UM = 1.25
RETAINED_MODES = 24
MONITOR_MODES = 24
PREFLIGHT_CASES = ("uniform",)  # Add "moving" after checking the corrected uniform launch.
SOURCE_KIND = "isolated"  # "mapped" uses the verified DREME input E/H
SOLVER_VERSION = None    # Tidy3D default; record actual version with the task
CLOUD_FOLDER = "DREME coupled SOI"


## 1 — Local preflight

Inspect the snapped plan and cross-section. The field source lies just before the input reference plane. Both reference planes store tangential E/H and direction-resolved mode profiles. Check DREME's input E/H residuals and P0/P1 before deciding the launch. Large launch mismatch after the initial run can be tested with `SOURCE_KIND="mapped"` and a new output root.

In [ ]:
cases = {}
for name in PREFLIGHT_CASES:
    case = prepare_case(
        name, DATASET, REFERENCE_FIELDS, OUTPUT_ROOT, coarse_dataset=COARSE_DATASET,
        lengths_um=LENGTHS_UM, mesh_um=FDTD_MESH_UM, run_time_ps=FDTD_RUN_TIME_PS,
        source_kind=SOURCE_KIND, source_x_um=SOURCE_X_UM, lead_um=LEAD_UM,
        retained_modes=RETAINED_MODES, monitor_modes=MONITOR_MODES,
    )
    cases[name] = case
    d = case.manifest["dreme"]
    print(name, "planes (m)", case.manifest["reference_planes_m"])
    print("Target Ey center y (um):", case.manifest["source"]["target_Ey_center_y_um"])
    print("DREME P0 / P1 / P2 / P3 (W):", [d["power"][f"P{i}"] for i in range(4)])
    print("Input relative E/H residual:", [d["input_direct_EH_residual"][f] for f in ("E", "H")])
    display(pd.DataFrame(case.dreme_rows))
    fig = plot_case(case)
    fig.savefig(case.directory / "preflight.png", dpi=150)
    display(fig)
    import matplotlib.pyplot as plt
    plt.close(fig)


## 2 — Upload draft and estimate maximum FlexCredits

Set `UPLOAD_CASE` to `"uniform"` first. This uploads a draft and estimates its maximum cost. The task ID is saved immediately; reruns reuse it. Repeat for `"moving"` after reviewing the first comparison. This cell does not start FDTD.

In [ ]:
UPLOAD_CASE = None  # "uniform" or "moving"
if UPLOAD_CASE is None:
    print("Set UPLOAD_CASE to the case you want to upload.")
else:
    task = upload_and_estimate(cases[UPLOAD_CASE], folder=CLOUD_FOLDER, solver_version=SOLVER_VERSION)
    print(UPLOAD_CASE, "| task ID:", task["task_id"], "| maximum FlexCredits:", task["max_flexcredits"])


## 3 — Run approved FDTD and save

Set `RUN_CASE` and paste that case's exact task ID in `APPROVED_TASK_ID` after reviewing its estimate. This is the only step that starts a new paid FDTD calculation. An already running task is monitored and downloaded, a successful task is downloaded, and an existing local result is opened directly.

In [ ]:
RUN_CASE = None  # "uniform" or "moving"
APPROVED_TASK_ID = ""
if RUN_CASE is None:
    print("Set RUN_CASE and APPROVED_TASK_ID after reviewing the estimate.")
else:
    run_or_download(cases[RUN_CASE], approved_task_id=APPROVED_TASK_ID)
    print("Saved", cases[RUN_CASE].directory / "result.hdf5")


## 4 — Compare saved results offline

Run after step 1 with any available local results, including after restarting the kernel. Use the normal output path or add an explicit local HDF5 path to `RESULT_FILES`. Check input launch residual, measured incident/reflected power, and raw E/H reconstruction before interpreting channel differences. The table reports absolute differences in percentage points.

In [ ]:
RESULT_FILES = {}  # optional: {"uniform": Path(r".../result.hdf5")}
for name, case in cases.items():
    result_path = Path(RESULT_FILES.get(name, case.directory / "result.hdf5"))
    if not result_path.exists():
        print("Awaiting", result_path)
        continue
    table, figure, diagnostic = compare_saved(case, result_path)
    print(name, "| incident and backward W:",
          diagnostic["input"]["forward_flux_W"], diagnostic["input"]["backward_flux_magnitude_W"])
    print("Raw input / target Ey center y (um):", diagnostic["input"]["raw_Ey_center_y_um"],
          diagnostic["input"]["target_Ey_center_y_um"])
    print("Reconstructed/native signed flux ratio (input / output):",
          diagnostic["input"]["reconstructed_over_native_signed_flux"],
          diagnostic["output"]["reconstructed_over_native_signed_flux"])
    print("Input target relative E/H residual:",
          diagnostic["input"]["target_EH_relative_residual_after_power_and_phase_alignment"])
    print("Raw input / output modal reconstruction:",
          diagnostic["input"]["raw_EH_reconstruction"], diagnostic["output"]["raw_EH_reconstruction"])
    display(table)
    display(figure)
    plt.close(figure)


## 5 — Audit completed local results and mathematical consistency

Run after section 0. This uses the six completed source-aligned cases listed in `audit_saved_results.py`. Confirm `SAVED_RUNS_ROOT` and choose an audit output folder. The audit rechecks all-node acquisition baselines, reloaded endpoint fields, basis invariance, mesh/time comparisons and a measured-input replay. The returned status records quantitative physical acceptance as open.


In [ ]:
from examples.coupled_waveguide.audit_saved_results import audit_saved_results

SAVED_RUNS_ROOT = DATA_ROOT
AUDIT_OUTPUT = DATA_ROOT / "commit6_offline_audit"
audit_table, audit_runs, math_checks = audit_saved_results(SAVED_RUNS_ROOT, AUDIT_OUTPUT)
print("Mathematical checks:", math_checks)
print("Quantitative physical accuracy acceptance remains open.")
print("Saved audit:", AUDIT_OUTPUT)
display(audit_table[audit_table.channel.isin(["bus_TE0", "ring_TE0"])])
